# 🩸 혈구 카운팅 — Colab 실험 노트북

런타임 → 런타임 유형 변경 → **T4 GPU** 를 고른 뒤 위에서부터 차례로 실행하세요. 전체 약 1시간 30분 예상.

| 단계 | 내용 | 예상 시간 |
|---|---|---|
| 실험 1 · 라벨 품질 (시드 3개) | 같은 BCCD 이미지 101장 × (원본 라벨 / 재라벨) × 시드 42·7·2024 | 약 15분 |
| 실험 2 · 본학습 | TXL-PBC 882장 × (YOLO11n / YOLO11s), 같은 100 epoch | 약 1시간 10분 |
| 결과 | 비교 표(README용), 웹 데모 내보내기, results.zip | 약 5분 |

평가는 모두 **TXL-PBC test (재라벨 정답)**, 누수 의심 이미지 제외. 개수 지표에는 95% 신뢰구간이 붙습니다.

> 무료 Colab은 오래 아무 입력이 없으면 연결이 끊길 수 있어요. 브라우저 탭을 열어 두고, 아래 Drive 저장 셀을 쓰면 중간 결과를 잃지 않습니다.

In [ ]:
# 1) 코드 가져오기 — 본인 GitHub 저장소 주소로 바꾸세요
REPO = "https://github.com/<YOUR_ID>/bloodcell-counter.git"
!git clone -q $REPO
%cd bloodcell-counter
!pip -q install -r requirements.txt

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 → 런타임 유형을 T4 GPU로 바꾸세요')

In [ ]:
# (선택) Google Drive에 중간 결과 저장 — 연결이 끊겨도 이미 끝난 실험은 남습니다
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/bloodcell-counter-results'
import os; os.makedirs(SAVE_DIR, exist_ok=True)

def backup():
    !rsync -a --include='*/' --include='best.pt' --include='results.*' --include='eval_summary.yaml' --include='*.png' --include='*.md' --exclude='*' runs/ {SAVE_DIR}/runs/
    print('Drive에 저장:', SAVE_DIR)

## 1. 데이터 준비 · EDA
BCCD(원본 라벨)와 TXL-PBC(재라벨)를 내려받아 클래스 순서를 RBC=0, WBC=1, Platelets=2로 맞춥니다.
EDA가 누수 의심 목록(`suspect_leaks.yaml`)을 만들고, 평가 때 자동으로 제외합니다.

In [ ]:
!python -m bloodcount.prepare_data
!python -m bloodcount.eda --dataset bccd > /dev/null
!python -m bloodcount.eda > /dev/null
!python -m bloodcount.make_label_ablation
from IPython.display import Image, display
display(Image('runs/eda/txl/relabel_comparison.png'))

## 2. 실험 1 — 라벨 품질 통제 실험 (시드 3개)
이미지·모델·설정이 같고 **라벨만 다른** 두 모델을 시드 3개로 반복합니다. 시드가 바뀌어도 차이가 유지되는지 확인하는 단계입니다.

In [ ]:
SEEDS = [42, 7, 2024]
for seed in SEEDS:
    for variant in ['bccd_labels', 'txl_labels']:
        name = f'abl_{variant}_s{seed}'
        !python -m bloodcount.train --config configs/train_ablation.yaml --data data/ablation/{variant}/data.yaml --name {name} --seed {seed}
        !python -m bloodcount.evaluate --weights runs/detect/{name}/weights/best.pt --name {name} > /dev/null
try: backup()
except NameError: pass
!python -m bloodcount.compare_runs --only abl_

## 3. 실험 2 — TXL-PBC 본학습
두 모델 모두 **같은 100 epoch · patience 20 · 시드 42** 로 학습해 모델 크기 효과만 비교합니다.

In [ ]:
!python -m bloodcount.train --config configs/train_baseline.yaml --name txl_yolo11n_s42 --seed 42
!python -m bloodcount.evaluate --weights runs/detect/txl_yolo11n_s42/weights/best.pt --name txl_yolo11n_s42 > /dev/null
try: backup()
except NameError: pass

In [ ]:
!python -m bloodcount.train --config configs/train.yaml --name txl_yolo11s_s42 --seed 42
!python -m bloodcount.evaluate --weights runs/detect/txl_yolo11s_s42/weights/best.pt --name txl_yolo11s_s42 > /dev/null
try: backup()
except NameError: pass

(선택, 시간이 남으면) 본학습도 시드를 늘려 보세요. 한 번에 약 40분씩 걸립니다.
```python
for seed in [7, 2024]:
    !python -m bloodcount.train --config configs/train.yaml --name txl_yolo11s_s{seed} --seed {seed}
    !python -m bloodcount.evaluate --weights runs/detect/txl_yolo11s_s{seed}/weights/best.pt --name txl_yolo11s_s{seed} > /dev/null
```

## 4. 결과 비교 — 이 표를 README에 붙이면 됩니다

In [ ]:
!python -m bloodcount.compare_runs
# 같은 test 이미지에서 짝지어 비교 (차이의 95% 신뢰구간이 0을 포함하면 차이를 말하기 어려움)
for seed in SEEDS:
    print(f'\n실험 1 · 시드 {seed}: 원본 라벨 vs 재라벨')
    !python -m bloodcount.compare_runs --paired abl_bccd_labels_s{seed} abl_txl_labels_s{seed}
print('\n실험 2: YOLO11n vs YOLO11s')
!python -m bloodcount.compare_runs --paired txl_yolo11n_s42 txl_yolo11s_s42
display(Image('runs/eval/txl_yolo11s_s42/threshold_curve.png'))
display(Image('runs/eval/txl_yolo11s_s42/count_scatter.png'))

## 5. 웹 데모 내보내기 · 결과 저장
가장 좋은 모델(기본: YOLO11s)을 브라우저 데모용으로 내보내고, 결과를 zip으로 내려받습니다.
YOLO11n이 더 나았다면 아래 `BEST`를 바꾸세요. 기준은 **smear RBC MAE**(서비스 대상인 넓은 시야 도말에서의 개수 오차)를 추천합니다.

In [ ]:
BEST = 'txl_yolo11s_s42'
!python -m bloodcount.export_web --weights runs/detect/{BEST}/weights/best.pt --from-eval runs/eval/{BEST}/eval_summary.yaml
!zip -qr results.zip runs/detect/*/weights/best.pt runs/detect/*/results.* runs/detect/*/args.yaml runs/eval web
import shutil
try:
    shutil.copy('results.zip', SAVE_DIR)
except NameError:
    pass
from google.colab import files
files.download('results.zip')